# Go2W Real Robot Deployment Tutorial: WiFi → Docker → Echo Topics

**A step-by-step guide to connecting your laptop to the Go2W over WiFi, containerizing the ROS2 driver stack with Docker, and verifying `cmd_vel` / `joint_states` topic visibility.**

---

## Table of Contents

1. [Network Architecture: How the Go2W Communicates](#1)
2. [WiFi Connection Setup](#2)
3. [CycloneDDS Configuration for WiFi](#3)
4. [Docker Containerization](#4)
5. [Building & Running the Container](#5)
6. [Echoing Topics: `cmd_vel` & `joint_states`](#6)
7. [Python: Programmatic Topic Monitor](#7)
8. [Exposed Interfaces Summary](#8)
9. [Troubleshooting](#9)
10. [Next Steps: Full Autonomy Stack](#10)

<a id='1'></a>
## 1. Network Architecture: How the Go2W Communicates

The Unitree Go2W has **two network interfaces** for external communication:

### Connection Options

In [ ]:
                         +============================+
                         |       GO2W ROBOT           |
                         |                            |
                         |  Onboard Jetson / MCU      |
                         |  DDS Domain 0              |
                         |                            |
                         |  WiFi AP: 192.168.12.1     |
                         |  Ethernet: 192.168.123.18  |
                         +====+================+======+
                              |                |
                 WiFi (wlan)  |                | Ethernet (cable)
                              |                |
                +-------------+--+      +------+-----------+
                |  Your Laptop   |  OR  |  Your Laptop     |
                | 192.168.12.x   |      | 192.168.123.100  |
                | (DHCP from AP) |      | (static)         |
                +----------------+      +------------------+

### WiFi vs Ethernet

| Aspect | WiFi (192.168.12.x) | Ethernet (192.168.123.x) |
|--------|--------------------|--------------------------|
| Setup | Join robot's WiFi AP | Plug cable into rear port |
| IP config | DHCP (automatic) | Static (manual) |
| Bandwidth | ~50 Mbps | ~1 Gbps |
| Latency | ~2-10 ms | <1 ms |
| Good for | Untethered testing, demos | High-bandwidth SLAM, LiDAR |
| Cable-free? | ✅ Yes | ❌ No |

### What the Robot Publishes (DDS Topics)

The Go2W's onboard stack publishes these topics over DDS:

| Topic | Type | Rate | Description |
|-------|------|------|-------------|
| `/utlidar/cloud` | PointCloud2 | ~10 Hz | L1 LiDAR point cloud |
| `/utlidar/robot_pose` | PoseStamped | ~50 Hz | LiDAR-based odometry |
| `lowstate` | unitree_go/LowState | ~500 Hz | Raw 16-motor joint states |
| `api/sport/request` | unitree_api/Request | (sub) | Motor commands input |

The **go2w_driver** node bridges these into standard ROS2 topics:

| Published Topic | Type | Description |
|----------------|------|-------------|
| `/joint_states` | sensor_msgs/JointState | 16 joints (12 leg + 4 wheel) |
| `/odom` | nav_msgs/Odometry | Wheel odometry |
| `/pointcloud` | sensor_msgs/PointCloud2 | Rebroadcast LiDAR |
| **Subscribed Topic** | **Type** | **Description** |
| `/cmd_vel` | geometry_msgs/Twist | **Velocity commands to robot** |

<a id='2'></a>
## 2. WiFi Connection Setup

### Step 1: Connect to the Go2W's WiFi Access Point

The Go2W broadcasts its own WiFi network. Join it like any other WiFi:

In [ ]:
# List available WiFi networks
nmcli device wifi list

# Connect (SSID is usually something like "Unitree_Go2WXXXXXX")
# The default password is printed on the robot or in Unitree docs
nmcli device wifi connect "Unitree_Go2WXXXXXX" password "YOUR_PASSWORD"

### Step 2: Verify IP assignment

In [ ]:
# Check your WiFi interface and assigned IP
ip addr show | grep -A2 'wl'

# You should see something like:
# wlan0: <BROADCAST,MULTICAST,UP> ...
#     inet 192.168.12.XXX/24 ...

### Step 3: Verify connectivity

In [ ]:
# Ping the robot's WiFi gateway
ping -c 3 192.168.12.1

# Expected: 3 replies, ~2-10ms RTT

### Step 4: Identify your WiFi interface name

In [ ]:
# Common names: wlan0, wlp0s20f3, wlp2s0
ip -br link show type wifi

# Note this name — you'll need it for CycloneDDS config

> **⚠️ Important**: If you also use Ethernet for internet (e.g., lab network), make sure DDS binds to the WiFi interface specifically (Section 3), otherwise ROS2 traffic may route through the wrong NIC.

<a id='3'></a>
## 3. CycloneDDS Configuration for WiFi

### Why CycloneDDS?

- The Go2W's onboard DDS stack uses **CycloneDDS**
- FastDDS (ROS2 Humble default) uses shared memory transport which doesn't work across machines
- CycloneDDS with explicit interface binding ensures all DDS discovery goes over WiFi

### Configuration

Create a file `cyclonedds_wifi.xml` or export the config inline:

In [ ]:
# Set the DDS middleware to CycloneDDS
export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp

# Tell CycloneDDS to use ONLY the WiFi interface
# ⚠️ Change 'wlan0' to YOUR WiFi interface name from Step 4 above!
export CYCLONEDDS_URI='<CycloneDDS><Domain><General><Interfaces>
    <NetworkInterface name="wlan0" priority="default" multicast="default" />
</Interfaces></General></Domain></CycloneDDS>'

### As a standalone XML file

```xml
<!-- cyclonedds_wifi.xml -->
<?xml version="1.0" encoding="UTF-8"?>
<CycloneDDS>
  <Domain>
    <General>
      <Interfaces>
        <!-- Change 'wlan0' to your WiFi interface name -->
        <NetworkInterface name="wlan0" priority="default" multicast="default" />
      </Interfaces>
    </General>
  </Domain>
</CycloneDDS>
```

Usage:

In [ ]:
export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp
export CYCLONEDDS_URI=file://$(pwd)/cyclonedds_wifi.xml

### Quick Verify

In [ ]:
# After exporting, you should see the robot's raw DDS topics:
ros2 topic list

# Expected (with driver NOT running — just raw robot topics):
# /utlidar/cloud
# /utlidar/robot_pose
# ... etc

<a id='4'></a>
## 4. Docker Containerization

### Why Docker?

- **Reproducible environment**: Same ROS2 Humble + CycloneDDS + Unitree dependencies everywhere
- **No host pollution**: Don't need to install `unitree_go`, `unitree_api`, etc. on your host
- **Easy deployment**: Build once, run on any Linux machine (your laptop, Jetson, cloud)
- **Isolation**: Container can't break your host ROS2 setup

### Architecture

In [ ]:
+===============================================================+
|                      YOUR LAPTOP                               |
|                                                                |
|  +----------------------------------------------------------+  |
|  |  Docker container (host network mode)                     |  |
|  |                                                           |  |
|  |  ros:humble-ros-base                                      |  |
|  |  + rmw_cyclonedds_cpp                                     |  |
|  |  + unitree_go  (msg defs)                                 |  |
|  |  + unitree_api (msg defs)                                 |  |
|  |  + go2_interfaces (srv defs)                              |  |
|  |  + go2w_driver (C++ node)                                 |  |
|  |  + go2w_description (URDF)                                |  |
|  |  + go2w_bringup (launch)                                  |  |
|  |                                                           |  |
|  |  Exposes:                                                 |  |
|  |    /cmd_vel        (Twist)       ← input                  |  |
|  |    /joint_states   (JointState)  → output                 |  |
|  |    /odom           (Odometry)    → output                 |  |
|  |    /pointcloud     (PointCloud2) → output                 |  |
|  +----------------------------------------------------------+  |
|          |  host network (--network=host)                      |
|          |  DDS multicast on WiFi interface                    |
+==========|=====================================================+
           |  WiFi: 192.168.12.x
      +----+---------------------------------------------+
      |                GO2W ROBOT                         |
      |  DDS Domain 0   192.168.12.1                     |
      +--------------------------------------------------+

### Key Docker decisions

1. **`--network=host`** — Required for DDS multicast discovery. Bridge/NAT networking breaks DDS.
2. **CycloneDDS bound to WiFi** — The XML config pins discovery to the WiFi NIC.
3. **Minimal image** — Only the driver + message definitions. No Gazebo, no SLAM, no navigation.

### Dockerfile

Create `docker/go2w_real/Dockerfile`:

```dockerfile
# ===========================================================================
# Stage 1: Build the go2w_driver and message packages from source
# ===========================================================================
FROM ros:humble-ros-base AS builder

SHELL ["/bin/bash", "-c"]

# Install build dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \
    git \
    python3-colcon-common-extensions \
    python3-rosdep \
    ros-humble-rmw-cyclonedds-cpp \
    nlohmann-json3-dev \
  && rm -rf /var/lib/apt/lists/*

# Create workspace
WORKDIR /ws/src

# Copy ONLY the packages we need (no Gazebo, no SLAM, no nav)
# -- Message definitions --
COPY src/unitree_go2w_ros2/src/unitree_go  unitree_go/
COPY src/unitree_go2w_ros2/src/unitree_api unitree_api/
COPY src/unitree_go2w_ros2/src/go2_interfaces go2_interfaces/
# -- Driver + description + bringup --
COPY src/unitree_go2w_ros2/src/go2w_driver go2w_driver/
COPY src/unitree_go2w_ros2/src/go2w_description go2w_description/
COPY src/unitree_go2w_ros2/src/go2w_bringup go2w_bringup/
COPY src/unitree_go2w_ros2/src/go2w_mock go2w_mock/
COPY src/unitree_go2w_ros2/src/go2_rviz go2_rviz/

# Build
WORKDIR /ws
RUN source /opt/ros/humble/setup.bash && \
    colcon build --symlink-install --cmake-args -DCMAKE_BUILD_TYPE=Release

# ===========================================================================
# Stage 2: Lean runtime image
# ===========================================================================
FROM ros:humble-ros-base AS runtime

SHELL ["/bin/bash", "-c"]

RUN apt-get update && apt-get install -y --no-install-recommends \
    ros-humble-rmw-cyclonedds-cpp \
  && rm -rf /var/lib/apt/lists/*

# Copy the built workspace from builder
COPY --from=builder /ws/install /ws/install

# Copy CycloneDDS config
COPY docker/go2w_real/cyclonedds_wifi.xml /ws/cyclonedds_wifi.xml

# Environment setup
ENV RMW_IMPLEMENTATION=rmw_cyclonedds_cpp
ENV CYCLONEDDS_URI=file:///ws/cyclonedds_wifi.xml
ENV ROS_DOMAIN_ID=0

# Entrypoint: source ROS2 + workspace overlay
COPY docker/go2w_real/entrypoint.sh /ws/entrypoint.sh
RUN chmod +x /ws/entrypoint.sh
ENTRYPOINT ["/ws/entrypoint.sh"]

# Default: launch the driver
CMD ["ros2", "launch", "go2w_bringup", "go2w.launch.py"]
```

### Entrypoint Script

Create `docker/go2w_real/entrypoint.sh`:

In [ ]:
#!/bin/bash
set -e

# Source ROS2 base
source /opt/ros/humble/setup.bash

# Source our workspace overlay
source /ws/install/setup.bash

# If WIFI_IFACE is set, rewrite the CycloneDDS config dynamically
if [ -n "${WIFI_IFACE}" ]; then
  cat > /ws/cyclonedds_wifi.xml <<EOF
<?xml version="1.0" encoding="UTF-8"?>
<CycloneDDS>
  <Domain>
    <General>
      <Interfaces>
        <NetworkInterface name="${WIFI_IFACE}" priority="default" multicast="default" />
      </Interfaces>
    </General>
  </Domain>
</CycloneDDS>
EOF
  echo "[entrypoint] CycloneDDS bound to interface: ${WIFI_IFACE}"
fi

# Execute the CMD
exec "$@"

### CycloneDDS WiFi Config

Create `docker/go2w_real/cyclonedds_wifi.xml`:

```xml
<?xml version="1.0" encoding="UTF-8"?>
<CycloneDDS>
  <Domain>
    <General>
      <Interfaces>
        <!-- Default: wlan0. Override at runtime with WIFI_IFACE env var -->
        <NetworkInterface name="wlan0" priority="default" multicast="default" />
      </Interfaces>
    </General>
  </Domain>
</CycloneDDS>
```

### docker-compose.yml

Create `docker/go2w_real/docker-compose.yml`:

```yaml
version: '3.8'

services:
  go2w_driver:
    build:
      context: ../..          # repo root (Dockerfile COPYs from src/)
      dockerfile: docker/go2w_real/Dockerfile
    network_mode: host        # required for DDS multicast
    environment:
      - RMW_IMPLEMENTATION=rmw_cyclonedds_cpp
      - CYCLONEDDS_URI=file:///ws/cyclonedds_wifi.xml
      - ROS_DOMAIN_ID=0
      - WIFI_IFACE=${WIFI_IFACE:-wlan0}
    # Default: launch driver
    command: ros2 launch go2w_bringup go2w.launch.py

  # Topic echo sidecar — runs alongside driver
  echo_topics:
    build:
      context: ../..          
      dockerfile: docker/go2w_real/Dockerfile
    network_mode: host
    environment:
      - RMW_IMPLEMENTATION=rmw_cyclonedds_cpp
      - CYCLONEDDS_URI=file:///ws/cyclonedds_wifi.xml
      - ROS_DOMAIN_ID=0
      - WIFI_IFACE=${WIFI_IFACE:-wlan0}
    depends_on:
      - go2w_driver
    command: >
      bash -c '
        source /opt/ros/humble/setup.bash &&
        source /ws/install/setup.bash &&
        echo "=== Waiting 5s for driver startup... ==" &&
        sleep 5 &&
        echo "" &&
        echo "============================================" &&
        echo "  TOPIC LIST" &&
        echo "============================================" &&
        ros2 topic list &&
        echo "" &&
        echo "============================================" &&
        echo "  /joint_states (one message)" &&
        echo "============================================" &&
        timeout 10 ros2 topic echo /joint_states --once || echo "(no message received within 10s)" &&
        echo "" &&
        echo "============================================" &&
        echo "  /joint_states frequency" &&
        echo "============================================" &&
        timeout 5 ros2 topic hz /joint_states || true &&
        echo "" &&
        echo "============================================" &&
        echo "  /cmd_vel info" &&
        echo "============================================" &&
        ros2 topic info /cmd_vel -v || echo "(topic not found — will appear when publisher connects)" &&
        echo "" &&
        echo "============================================" &&
        echo "  ✅ VERIFICATION COMPLETE" &&
        echo "  Interfaces exposed:" &&
        echo "    → /cmd_vel       (geometry_msgs/Twist) — SEND commands here" &&
        echo "    ← /joint_states  (sensor_msgs/JointState) — READ joint data here" &&
        echo "    ← /odom          (nav_msgs/Odometry) — READ odometry here" &&
        echo "============================================"
      '
```

> **Note**: The `echo_topics` service runs the verification once and exits. The `go2w_driver` service keeps running.

<a id='5'></a>
## 5. Building & Running the Container

### Prerequisites

In [ ]:
# Install Docker if needed
# https://docs.docker.com/engine/install/ubuntu/
docker --version  # should be 20.10+
docker compose version  # should be v2+

### Step 1: Identify your WiFi interface

In [ ]:
# Find your WiFi interface name
WIFI_IFACE=$(ip -br link show type wifi | awk 'NR==1{print $1}')
echo "WiFi interface: $WIFI_IFACE"

### Step 2: Build

In [ ]:
cd ~/COMP0225_LRC_stack

# Build the Docker image (first time takes ~5-10 min)
docker compose -f docker/go2w_real/docker-compose.yml build

### Step 3: Run

In [ ]:
# Launch driver + echo verification
WIFI_IFACE=$WIFI_IFACE docker compose -f docker/go2w_real/docker-compose.yml up

# The echo_topics service will:
# 1. Wait 5 seconds for the driver to start
# 2. List all visible topics
# 3. Echo one /joint_states message
# 4. Measure /joint_states frequency
# 5. Show /cmd_vel topic info
# 6. Print a summary of exposed interfaces

### Step 4: Verify from host (separate terminal)

In [ ]:
# In a new terminal on your laptop (NOT in Docker):
source /opt/ros/humble/setup.bash
export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp
export CYCLONEDDS_URI='<CycloneDDS><Domain><General><Interfaces>
    <NetworkInterface name="'$WIFI_IFACE'" priority="default" multicast="default" />
</Interfaces></General></Domain></CycloneDDS>'

# You should see topics from both the container AND the robot:
ros2 topic list

# Echo joint states
ros2 topic echo /joint_states --once

# Send a test cmd_vel (ZERO velocity — safe)
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}" --once

### Stopping

In [ ]:
# Ctrl+C in the docker compose terminal, then:
WIFI_IFACE=$WIFI_IFACE docker compose -f docker/go2w_real/docker-compose.yml down

<a id='6'></a>
## 6. Echoing Topics: `cmd_vel` & `joint_states`

### Understanding `/joint_states`

The Go2W driver publishes 16 joint positions in this order:

| Index | Joint Name | Motor Index | Type |
|-------|------------|-------------|------|
| 0 | `FL_hip_joint` | 3 | Leg |
| 1 | `FL_thigh_joint` | 4 | Leg |
| 2 | `FL_calf_joint` | 5 | Leg |
| 3 | `FL_foot_joint` | **12** | **Wheel** |
| 4 | `FR_hip_joint` | 0 | Leg |
| 5 | `FR_thigh_joint` | 1 | Leg |
| 6 | `FR_calf_joint` | 2 | Leg |
| 7 | `FR_foot_joint` | **13** | **Wheel** |
| 8 | `RL_hip_joint` | 9 | Leg |
| 9 | `RL_thigh_joint` | 10 | Leg |
| 10 | `RL_calf_joint` | 11 | Leg |
| 11 | `RL_foot_joint` | **14** | **Wheel** |
| 12 | `RR_hip_joint` | 6 | Leg |
| 13 | `RR_thigh_joint` | 7 | Leg |
| 14 | `RR_calf_joint` | 8 | Leg |
| 15 | `RR_foot_joint` | **15** | **Wheel** |

### Echo from command line

In [ ]:
# Full joint_states message
ros2 topic echo /joint_states --once

# Just the joint names
ros2 topic echo /joint_states --field name --once

# Joint positions only
ros2 topic echo /joint_states --field position --once

# Continuous monitoring at reduced rate
ros2 topic echo /joint_states --field position --no-arr

### Understanding `/cmd_vel`

In [ ]:
geometry_msgs/msg/Twist:
  linear:
    x: float  → forward/backward (m/s)     positive = forward
    y: float  → left/right strafe (m/s)    positive = left  
    z: float  → (unused for ground robot)
  angular:
    x: float  → (unused)
    y: float  → (unused)
    z: float  → yaw rotation (rad/s)       positive = counter-clockwise

In [ ]:
# Echo cmd_vel to see what's being sent to the robot
ros2 topic echo /cmd_vel

# Check the topic info (publishers/subscribers)
ros2 topic info /cmd_vel -v

<a id='7'></a>
## 7. Python: Programmatic Topic Monitor

Run this from your laptop (with CycloneDDS sourced) to continuously monitor the two key interfaces:

In [ ]:
#!/usr/bin/env python3
"""Monitor /joint_states and /cmd_vel from a Go2W robot over WiFi.

Prerequisites:
  - Connected to Go2W WiFi (192.168.12.x)
  - CycloneDDS configured (see tutorial Section 3)
  - go2w_driver running (either natively or in Docker container)
"""

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
from geometry_msgs.msg import Twist
import time


class Go2WTopicMonitor(Node):
    def __init__(self):
        super().__init__('go2w_topic_monitor')

        # --- Joint States subscriber ---
        self.joint_state_sub = self.create_subscription(
            JointState, '/joint_states', self.joint_state_cb, 10)
        self.joint_count = 0
        self.joint_first_time = None
        self.last_joint_msg = None

        # --- cmd_vel subscriber ---
        self.cmd_vel_sub = self.create_subscription(
            Twist, '/cmd_vel', self.cmd_vel_cb, 10)
        self.cmd_count = 0

        # Status printer
        self.timer = self.create_timer(2.0, self.print_status)
        self.get_logger().info('Monitoring /joint_states and /cmd_vel...')
        self.get_logger().info('(Waiting for messages...)')

    def joint_state_cb(self, msg: JointState):
        if self.joint_first_time is None:
            self.joint_first_time = time.monotonic()
            self.get_logger().info(
                f'✅ /joint_states CONNECTED! {len(msg.name)} joints: '
                f'{", ".join(msg.name[:4])}...')
        self.joint_count += 1
        self.last_joint_msg = msg

    def cmd_vel_cb(self, msg: Twist):
        self.cmd_count += 1
        self.get_logger().info(
            f'📡 /cmd_vel received: '
            f'linear=({msg.linear.x:.3f}, {msg.linear.y:.3f}) '
            f'angular.z={msg.angular.z:.3f}')

    def print_status(self):
        # Joint states rate
        if self.joint_first_time and self.joint_count > 1:
            elapsed = time.monotonic() - self.joint_first_time
            hz = self.joint_count / elapsed if elapsed > 0 else 0
            # Show wheel positions (indices 3, 7, 11, 15)
            wheel_pos = ''
            if self.last_joint_msg and len(self.last_joint_msg.position) >= 16:
                wheels = [self.last_joint_msg.position[i] for i in (3, 7, 11, 15)]
                wheel_pos = f' | wheels: [{", ".join(f"{w:.2f}" for w in wheels)}]'
            self.get_logger().info(
                f'📊 /joint_states: {hz:.1f} Hz ({self.joint_count} msgs){wheel_pos}')
        elif self.joint_count == 0:
            self.get_logger().warn('⏳ /joint_states: no messages yet')

        # cmd_vel stats
        if self.cmd_count > 0:
            self.get_logger().info(f'📊 /cmd_vel: {self.cmd_count} msgs received total')


def main():
    rclpy.init()
    node = Go2WTopicMonitor()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info('Shutting down monitor.')
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()

### Running the monitor

In [ ]:
# Save the above cell as monitor_go2w.py, then:
python3 monitor_go2w.py

# Expected output:
# [INFO] Monitoring /joint_states and /cmd_vel...
# [INFO] ✅ /joint_states CONNECTED! 16 joints: FL_hip_joint, FL_thigh_joint, FL_calf_joint, FL_foot_joint...
# [INFO] 📊 /joint_states: 50.2 Hz (100 msgs) | wheels: [0.01, -0.02, 0.00, 0.01]

<a id='8'></a>
## 8. Exposed Interfaces Summary

After running the Docker container, these interfaces are available on the DDS network for any ROS2 node (inside or outside the container) to use:

### Input (send commands TO the robot)

| Topic | Type | Rate | Description |
|-------|------|------|-------------|
| `/cmd_vel` | `geometry_msgs/Twist` | Publish at ≥10 Hz | Forward/turn velocity commands |

### Output (read data FROM the robot)

| Topic | Type | Rate | Description |
|-------|------|------|-------------|
| `/joint_states` | `sensor_msgs/JointState` | ~50 Hz | 16 joint positions (12 leg + 4 wheel) |
| `/odom` | `nav_msgs/Odometry` | ~50 Hz | Wheel odometry (position + orientation) |
| `/pointcloud` | `sensor_msgs/PointCloud2` | ~10 Hz | L1 LiDAR point cloud |
| `/imu` | `unitree_go/IMUState` | ~50 Hz | Onboard IMU |

### Services (configure robot behavior)

| Service | Type | Description |
|---------|------|-------------|
| `/mode` | `go2_interfaces/srv/Mode` | Stand up / sit / damp / tricks |
| `/switch_joystick` | `go2_interfaces/srv/SwitchJoystick` | Enable/disable physical remote |
| `/switch_gait` | `go2_interfaces/srv/SwitchGait` | 0=Move, 1=Terrain, 2=Climb |
| `/body_height` | `go2_interfaces/srv/BodyHeight` | Adjust standing height |
| `/speed_level` | `go2_interfaces/srv/SpeedLevel` | -1=Low, 0=Normal, 1=High |

### How the autonomy stack will connect (future)

In [ ]:
Docker Container (driver)          Your Autonomy Code (host or another container)
+----------------------------+     +-------------------------------------------+
|  go2w_driver               |     |  SLAM (Point-LIO)                         |
|  /joint_states  ──────────────>  |  /joint_states → robot_state_publisher    |
|  /pointcloud    ──────────────>  |  /pointcloud → SLAM → /odom/nav          |
|  /odom          ──────────────>  |                                           |
|  /cmd_vel       <────────────────  |  reactive_nav → twist_bridge → /cmd_vel  |
+----------------------------+     +-------------------------------------------+

<a id='9'></a>
## 9. Troubleshooting

### "Docker build fails with missing package"

The Dockerfile COPYs packages from `src/unitree_go2w_ros2/src/`. Make sure you've cloned all submodules:

In [ ]:
cd ~/COMP0225_LRC_stack
git submodule update --init --recursive

### "No topics visible from container"

1. **Check network mode**: Must be `--network=host`. Verify:

In [ ]:
   docker inspect <container_id> | grep NetworkMode
   # Should show: "NetworkMode": "host"
   

2. **Check CycloneDDS interface**: The XML must reference the correct WiFi interface:

In [ ]:
   # Inside the container:
   docker exec -it <container_id> cat /ws/cyclonedds_wifi.xml
   # Verify the NetworkInterface name matches your WiFi
   

3. **Check ROS_DOMAIN_ID**: Must be 0 (default) for both container and robot:

In [ ]:
   echo $ROS_DOMAIN_ID  # should be 0 or unset
   

### "I see topics but /joint_states has no data"

- The robot must be powered on and the onboard DDS stack running
- Check that `lowstate` topic (raw) is being published by the robot:

In [ ]:
  ros2 topic hz lowstate
  

### "/cmd_vel is published but robot doesn't move"

1. **Call `/switch_joystick`** first to disable the physical remote:

In [ ]:
   ros2 service call /switch_joystick go2_interfaces/srv/SwitchJoystick "{flag: false}"
   

2. **Call `/mode stand_up`** to put robot in walking posture:

In [ ]:
   ros2 service call /mode go2_interfaces/srv/Mode "{mode: 'stand_up'}"
   

3. Make sure you're publishing at **≥10 Hz** (not just `--once`)

### "Container can see host topics but not robot topics"

- WiFi may have poor multicast support. Try:

In [ ]:
  # Disable multicast, use unicast peer list instead:
  export CYCLONEDDS_URI='<CycloneDDS><Domain><General><Interfaces>
      <NetworkInterface name="wlan0" priority="default" multicast="false" />
  </Interfaces></General>
  <Discovery>
      <Peers><Peer address="192.168.12.1" /></Peers>
      <ParticipantIndex>auto</ParticipantIndex>
  </Discovery>
  </Domain></CycloneDDS>'
  

### WiFi performance issues

- Point cloud data (~10 Hz, ~1MB/msg) can saturate WiFi. Consider:
  - Using Ethernet for high-bandwidth data (SLAM), WiFi for cmd_vel only
  - Downsampling point clouds before transmission
  - Running SLAM on the robot's onboard Jetson, only transmitting odometry over WiFi

<a id='10'></a>
## 10. Next Steps: Full Autonomy Stack

Today we established the **foundation**: network → Docker → topic visibility.

### What we accomplished (Phase 1)

- [x] WiFi connection to Go2W
- [x] CycloneDDS configuration for WiFi interface
- [x] Docker container with go2w_driver
- [x] Verified `/cmd_vel` (input) and `/joint_states` (output) interfaces

### Upcoming phases

| Phase | Goal | What to add |
|-------|------|-------------|
| 2 | SLAM over WiFi | Add Point-LIO to container, verify `/odom/nav` |
| 3 | Mapping | Add `pointcloud_to_laserscan` + `simple_scan_mapper_cpp` |
| 4 | Autonomous exploration | Add CFPA2 + reactive_nav |
| 5 | Full stack | Use `single_go2w_real_cfpa2.launch.py` in Docker |

### The real-robot launch already exists!

In [ ]:
# When you're ready for the full stack:
ros2 launch go2_gazebo_sim single_go2w_real_cfpa2.launch.py \
  robot_namespace:=robot \
  rviz:=true \
  enable_manual_fallback:=true

See `tutorial_cmd_vel_go2.ipynb` Section 9 for details on the full stack.

---

## Quick Reference Card

In [ ]:
# === CONNECT TO ROBOT WiFi ===
nmcli device wifi connect "Unitree_Go2WXXXXXX" password "YOUR_PASSWORD"

# === FIND WiFi INTERFACE ===
WIFI_IFACE=$(ip -br link show type wifi | awk 'NR==1{print $1}')

# === DOCKER BUILD + RUN ===
cd ~/COMP0225_LRC_stack
docker compose -f docker/go2w_real/docker-compose.yml build
WIFI_IFACE=$WIFI_IFACE docker compose -f docker/go2w_real/docker-compose.yml up

# === VERIFY FROM HOST ===
export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp
export CYCLONEDDS_URI='<CycloneDDS><Domain><General><Interfaces>
    <NetworkInterface name="'$WIFI_IFACE'" priority="default" multicast="default" />
</Interfaces></General></Domain></CycloneDDS>'
ros2 topic list
ros2 topic echo /joint_states --once
ros2 topic echo /cmd_vel

# === ENABLE ROBOT + TEST CMD_VEL ===
ros2 service call /switch_joystick go2_interfaces/srv/SwitchJoystick "{flag: false}"
ros2 service call /mode go2_interfaces/srv/Mode "{mode: 'stand_up'}"
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.1, y: 0, z: 0}, angular: {x: 0, y: 0, z: 0}}" --rate 10

# === STOP ===
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0, y: 0, z: 0}, angular: {x: 0, y: 0, z: 0}}" --once